In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import BaseEstimator,TransformerMixin

In [ ]:
class IQROutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bounds_ = {}
        self.upper_bounds_ = {}

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        
        q25 = X_df.quantile(0.25)
        q75 = X_df.quantile(0.75)
        iqr = q75 - q25
        
        for col in X_df.columns:
            self.lower_bounds_[col] = q25[col] - (self.factor * iqr[col])
            self.upper_bounds_[col] = q75[col] + (self.factor * iqr[col])
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        
        for col in X_df.columns:
            if col in self.lower_bounds_:
                X_df[col] = np.clip(X_df[col], self.lower_bounds_[col], self.upper_bounds_[col])
        
        return X_df.to_numpy() if isinstance(X, np.ndarray) else X_df

In [ ]:
df=pd.read_csv('Titanic-Dataset.csv')

In [ ]:
df['Family']=df['Parch']+df['SibSp']+1

In [ ]:
columns_to_drop=['PassengerId','Name','Ticket','Cabin','Parch','SibSp']

In [ ]:
new_df=df.drop(columns=columns_to_drop)

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(new_df.drop(columns=['Survived']),new_df['Survived'],random_state=42,test_size=0.10)

In [ ]:
ct1=ColumnTransformer([
    ('impute_age',SimpleImputer(strategy='median'),[2]),
    ('impute_embarked',SimpleImputer(strategy='most_frequent'),[4])
    
],remainder='passthrough')

In [ ]:
ct2=ColumnTransformer([
    ('OHE_sex',OneHotEncoder(dtype=np.int32),[3]),
    ('OHE_embarked',OneHotEncoder(dtype=np.int32),[1])
],remainder='passthrough')

In [ ]:
ct3=ColumnTransformer([
    ('Capping_outlier',IQROutlierCapper(factor=1.5),[5,7])
],remainder='passthrough')

In [ ]:
pipe=Pipeline([
    ('ct1',ct1),
    ('ct2',ct2),
    ('ct3',ct3)
])

In [ ]:
pipe.fit(x_train)

In [ ]:
x_train_transform=pipe.transform(x_train)

In [ ]:
x_test_transform = pipe.transform(x_test)

In [ ]:
dt=DecisionTreeClassifier(criterion='gini',max_depth=8,min_samples_split=9,min_samples_leaf=5)

In [ ]:
dt_trained=dt.fit(x_train_transform,y_train)

In [ ]:
y_pred=dt_trained.predict(x_test_transform)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import accuracy_score
accuracy=accuracy_score(y_test,y_pred)

In [ ]:
accuracy

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))